# Combo A: Full Disease Investigation PipelineBuild organism -> equilibrium -> apply disease -> detect symptoms -> diagnose -> cure -> verify recovery.

In [ ]:
import sys, ossys.path.insert(0, os.path.join(os.getcwd(), "demos"))from _shared import make_homeostatic_system, make_disease_system, oracle_agent, random_agentfrom alienbio.bio import AgentInterface, DiagnoseTask, CureTask, check_stability, detect_symptomsfrom alienbio.viz import concentration_trajectory, symptom_chart

## Phase 1: Healthy Baseline

In [ ]:
system, baseline, perturbations = make_disease_system(seed=42)healthy_system = make_homeostatic_system(seed=42)healthy_tl = healthy_system.run(steps=500)stability = check_stability(healthy_tl, window=100, threshold=1e-4)print(f"Stable: {stability.stable}")for mol, conc in baseline.steady_state.items():    print(f"  {mol}: {conc:.4f}")

In [ ]:
concentration_trajectory(healthy_tl, title="Healthy Baseline")

## Phase 2: Apply Disease

In [ ]:
disease = perturbations[0]print(f"Disease: {disease.name}")disease.apply(system)diseased_tl = system.run(steps=300)

In [ ]:
concentration_trajectory(diseased_tl, title="Diseased System")

## Phase 3: Detect Symptoms

In [ ]:
concentrations = {name: system.state[name] for name in system.chemistry.molecules}symptoms = detect_symptoms(concentrations, baseline)for s in symptoms:    print(f"{s.molecule}: {s.value:.4f} (deviation={s.deviation:.4f})")

In [ ]:
symptom_chart(symptoms, baseline, title="Symptoms")

## Phase 4: Diagnosis

In [ ]:
task = DiagnoseTask(perturbations, applied_index=0)iface = AgentInterface(system)print(f"Oracle: {task.score(iface, oracle_agent(iface, task)).score:.2f}")print(f"Random: {task.score(iface, random_agent(iface, task)).score:.2f}")

## Phase 5: Cure

In [ ]:
cure_task = CureTask(baseline, recovery_steps=500)cured = make_homeostatic_system(seed=42)cured.run(steps=500)result = cure_task.score(AgentInterface(cured))print(f"Cure score: {result.score:.2f}")